# Week 31, Day 1: Reducing Hodgkin-Huxley: two variables and the Morris-Lecar model

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How much faster is m?

Compute the time constants of m, h and n for voltages from -80 to +20 mV, store them in the dictionary `tau`, and print the smallest ratio of the h and n time constants to that of m.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rates(V):
    V = V + 1e-7
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return am, bm, ah, bh, an, bn

def inf_tau(V, g):
    am, bm, ah, bh, an, bn = rates(V)
    a, b = {"m": (am, bm), "h": (ah, bh), "n": (an, bn)}[g]
    return a / (a + b), 1 / (a + b)

V = np.arange(-80, 21, 1.0)
tau = {g: ... for g in "mhn"}
print(f"tau_m from {tau['m'].min():.2f} to {tau['m'].max():.2f} ms; smallest ratio tau_h/tau_m {np.min(tau['h'] / tau['m']):.1f}, tau_n/tau_m {np.min(tau['n'] / tau['m']):.1f}")
for g in "mhn":
    plt.plot(V, tau[g], label=f"tau_{g}")
plt.xlabel("V (mV)"); plt.ylabel("ms"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_taus():
    "tau holds the three time constants at each voltage"
    assert all(np.allclose(tau[g], inf_tau(V, g)[1]) for g in "mhn")

def test_ratio():
    "m is at least three times faster than both h and n everywhere between -80 and +20 mV"
    assert np.all(tau["h"] / tau["m"] > 3) and np.all(tau["n"] / tau["m"] > 3)

test_taus()
test_ratio()
print("All checks passed")

### Exercise: Fitting h as a function of n

`full(I)` simulates the full model and returns arrays of V, n and h. Run it at 10 μA/cm² for 200 ms, take n and h after the first 50 ms (`nn`, `hh`), and fit h = a + b n by least squares.

In [ ]:
def full(I, T=200.0, dt=0.01):
    V, n, m, h = -65.0, 0.3177, 0.0529, 0.5961
    out = np.empty((int(T / dt), 3))
    for k in range(len(out)):
        am, bm, ah, bh, an, bn = rates(V)
        dV = I - 120 * m**3 * h * (V - 50) - 36 * n**4 * (V + 77) - 0.3 * (V + 54.387)
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        V += dt * dV
        out[k] = V, n, h
    return out

o = full(10.0)
nn, hh = o[5000:, 1], o[5000:, 2]
b, a = ...
print(f"h = {a:.3f} + ({b:.3f}) n;  correlation {np.corrcoef(nn, hh)[0, 1]:.4f}")
plt.plot(nn, hh, ".", ms=1); plt.plot(nn, a + b * nn); plt.xlabel("n"); plt.ylabel("h"); plt.show()

In [ ]:
import numpy as np
def test_fit():
    "a and b are the intercept and slope of the least-squares line of h on n"
    p = np.polyfit(nn, hh, 1)
    assert abs(b - p[0]) < 1e-9 and abs(a - p[1]) < 1e-9

def test_tight():
    "the relation is tight: correlation below -0.98, with slope near -1"
    assert np.corrcoef(nn, hh)[0, 1] < -0.98 and -1.3 < b < -0.8

test_fit()
test_tight()
print("All checks passed")

### Exercise: How good is the reduction?

Complete `reduced(I)` using m∞(V) and h = a + b n, write `rate(v)` (mean rate from upward crossings of 0 mV after 50 ms, with steps of 0.01 ms), and compare the firing rates of the full and reduced models at 5, 7, 10, 20 and 40 μA/cm².

In [ ]:
def reduced(I, T=200.0, dt=0.01):
    V, n = -65.0, 0.3177
    out = np.empty(int(T / dt))
    for k in range(len(out)):
        am, bm, ah, bh, an, bn = rates(V)
        m, h = ..., ...
        dV = I - 120 * m**3 * h * (V - 50) - 36 * n**4 * (V + 77) - 0.3 * (V + 54.387)
        n += dt * (an * (1 - n) - bn * n)
        V += dt * dV
        out[k] = V
    return out

def rate(v, dt=0.01, t0=50.0):
    s = np.where((v[1:] > 0) & (v[:-1] <= 0))[0] * dt
    s = s[s > t0]
    return ...

currents = (5, 7, 10, 20, 40)
f_full = {I: rate(full(I)[:, 0]) for I in currents}
f_red = {I: rate(reduced(I)) for I in currents}
for I in currents:
    print(f"I = {I:2d}: full {f_full[I]:6.1f} Hz, reduced {f_red[I]:6.1f} Hz")

In [ ]:
import numpy as np
def test_rate():
    "rate counts upward crossings of 0 mV after 50 ms and converts the mean interval to Hz"
    v = np.zeros(20001); v[[6000, 8000, 10000]] = 10
    assert abs(rate(v) - 50) < 1e-6

def test_compare():
    "both models fire at 10 uA/cm2, and the reduced model fires faster than the full one at every current that makes both fire"
    assert f_full[10] > 0 and f_red[10] > 0
    assert all(f_red[I] > f_full[I] for I in currents if f_full[I] > 0)

test_rate()
test_compare()
print("All checks passed")

### Exercise: Two classes in one model

Complete the Morris-Lecar right-hand side in `ml_rate(I, cls)`, which returns the firing rate after 300 ms. Then find, for each parameter set, the lowest rate you can get by turning up the current from below threshold.

In [ ]:
import math

def ml_rate(I, cls, T=1000.0, dt=0.05):
    C, gCa, gK, gL, VCa, VK, VL, V1, V2 = 20.0, 4.4, 8.0, 2.0, 120.0, -84.0, -60.0, -1.2, 18.0
    if cls == 2:
        V3, V4, phi = 2.0, 30.0, 0.04
    else:
        V3, V4, phi, gCa = 12.0, 17.4, 1 / 15, 4.0
    V, w, spikes = -60.0, 0.0, []
    for k in range(int(T / dt)):
        minf = 0.5 * (1 + math.tanh((V - V1) / V2))
        winf = 0.5 * (1 + math.tanh((V - V3) / V4))
        tw = 1 / math.cosh((V - V3) / (2 * V4))
        dV = ...
        w += dt * phi * (winf - w) / tw
        Vnew = V + dt * dV
        if Vnew > 0 >= V and k * dt > 300:
            spikes.append(k * dt)
        V = Vnew
    return 1000 * (len(spikes) - 1) / (spikes[-1] - spikes[0]) if len(spikes) > 1 else 0.0

for cls, Is in ((2, (80, 85, 88, 89, 90, 100)), (1, (39.8, 40, 40.5, 42, 50, 80))):
    print(f"set {cls}:", ", ".join(f"I={I}: {ml_rate(I, cls, T=3000 if cls == 1 else 1000):.1f} Hz" for I in Is))

In [ ]:
import numpy as np
def test_class2():
    "with the first set, the model is silent at 85 uA/cm2 and jumps to a rate above 8 Hz at 90"
    assert ml_rate(85, 2) == 0 and ml_rate(90, 2) > 8

def test_class1():
    "with the second set, the model can fire below 3 Hz near its threshold"
    assert 0 < ml_rate(40.2, 1, T=3000) < 3

test_class2()
test_class1()
print("All checks passed")